In [1]:
from dotenv import load_dotenv
load_dotenv()

import langchain
#import langgraph
print(f"LangChain: {langchain.__version__}")
#print(f"LangGraph: {langgraph.__version__}")

LangChain: 1.3.11


In [2]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model="gpt-oss:120b-cloud", temperature=0)

# ── Alternatively, use Groq ──────────────────────────────────────────────────
# from langchain_groq import ChatGroq
# llm = ChatGroq(model="llama-3.3-70b-versatile")
# ─────────────────────────────────────────────────────────────────────────────

print("LLM ready.")

LLM ready.


In [3]:
from langchain_core.tools import tool

# Reusing the familiar tools from previous lessons
@tool
def get_weather(city: str) -> str:
    """Returns the current weather for a given city."""
    return f"The weather in {city} is sunny with a high of 28°C."

@tool
def get_stock_price(ticker: str) -> str:
    """Returns the current stock price for a given ticker symbol.
    Use 'ZENSAR' for Zensar Technologies, 'GOOGL' for Google."""
    prices = {"ZENSAR": "464.00 INR", "GOOGL": "175.00 USD"}
    return prices.get(ticker.upper(), f"Unknown ticker: {ticker}")

print("Tools ready: get_weather, get_stock_price")

Tools ready: get_weather, get_stock_price


In [4]:
from langchain.agents import create_agent

# No checkpointer → no memory
agent_no_memory = create_agent(
    model=llm,
    tools=[get_weather, get_stock_price],
    system_prompt="You are a helpful assistant.",
)

print("Agent created (no memory).")

Agent created (no memory).


MemorySaver lives in RAM — restart Python and everything is lost.
For production, you need disk persistence.

SqliteSaver writes every checkpoint to a .db file.
Stop the server, restart it, and the conversation picks up exactly where it left off.

In [5]:
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver
from langchain.agents import create_agent

# ✅ Lets connect to SQLite databse file
db_path = "conversation_memory.db"

conn = sqlite3.connect(db_path, check_same_thread=False)
sqlite_checkpointer = SqliteSaver(conn)

# MODEL AGENT CREATED

agent_persistent = create_agent(
    model=llm,
    tools=[get_weather, get_stock_price],
    system_prompt="You are a helpful assistant.",

    # ✅ New line for SQLite checkpointer
    checkpointer=sqlite_checkpointer,
)

print(f"Agent with SQLite checkpointer ready -> '{db_path}'")

Agent with SQLite checkpointer ready -> 'conversation_memory.db'


In [ ]:
#User Logs and has a conversion

config_persistent = {"configurable": {"thread_id": "persistent_user_1"}}

print("===SESSION 1===")
print()

r1 = agent_persistent.invoke(
    {"messages": [("user","Hi! My name is Sumit. I prefer weather updates in Celsius.")]},
    config=config_persistent
)

print(f"Turn 1 - Agent: {r1['messages'][-1].content}")

print()

r2 = agent_persistent.invoke(
    {"messages": [("user", "What is the weather in Mumbai?")]},
    config=config_persistent
)
print(f"Turn 2 — Agent: {r2['messages'][-1].content}")

print()
print("Imagine the server restarts now...")
print("(In a real scenario you would restart the kernel here)")

This is scenario 2, if User come after restart

In [9]:
config_persistent = {"configurable": {"thread_id": "persistent_user_1"}}

print("===SESSION 2 (resuming same thread_id) ===")
print()

r3 = agent_persistent.invoke(
    {"messages": [("user", "What is my name again? And what's the Zensar stock price?")]},
    config=config_persistent  #Same thread_id as before
)
print(f"Turn 3 - Agent: {r3['messages'][-1].content}")
print()

print("✅ The agent remembers Sumit's name from Session 1 — data was on disk.")


===SESSION 2 (resuming same thread_id) ===

Turn 3 - Agent: Your name is **Sumit**.  

The current stock price for **Zensar Technologies (ZENSAR)** is **₹ 464.00**. Let me know if you need any more information!

✅ The agent remembers Sumit's name from Session 1 — data was on disk.
